In [29]:
import numpy as np
import pandas as pd
import torch
import os
import PIL.Image as Image
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

In [2]:
images_path = "data/CBIS/jpeg/"
print(len(os.listdir(images_path)))

6774


In [3]:
df = pd.read_csv("data/CBIS/csv/mass_case_description_train_set.csv")
print(df.columns[11])
path = (df[df.columns[11]][0].split("/")[2])
print(images_path + path)
print(os.listdir(images_path + path))

image file path
data/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515
['1-211.jpg']


In [ ]:
for i in range(df.shape[0]):
    path = (df[df.columns[11]][i].split("/")[2])
    print(path)

1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515
1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834
1.3.6.1.4.1.9590.100.1.2.89180046211022531834352631483669346540
1.3.6.1.4.1.9590.100.1.2.295360926313492745441868049270168300162
1.3.6.1.4.1.9590.100.1.2.410524754913057908920631336070876889890
1.3.6.1.4.1.9590.100.1.2.392091931911637760938815694332198115839
1.3.6.1.4.1.9590.100.1.2.18553022011298363903753970133853455201
1.3.6.1.4.1.9590.100.1.2.399831242111800621220027542190666363688
1.3.6.1.4.1.9590.100.1.2.353764633213863442442494539840710764601
1.3.6.1.4.1.9590.100.1.2.301773695910378556700140939623830965391
1.3.6.1.4.1.9590.100.1.2.145557498613842583714429986600518227573
1.3.6.1.4.1.9590.100.1.2.369287458813355648611939058220166307923
1.3.6.1.4.1.9590.100.1.2.238517328812179290330581800221092807231
1.3.6.1.4.1.9590.100.1.2.84350006812506903321489911520378117746
1.3.6.1.4.1.9590.100.1.2.123968816011512951434814084172747879091
1.3.6.1.4.1.9590.100.1.2.417

In [7]:
def split_data(data):
    return torch.utils.data.random_split(data, [0.8, 0.2])

In [8]:
def preprocess_image(image):
    # Resize the image to a fixed size (e.g., 224x224)
    image = image.resize((224, 224))
    
    # Convert the image to a tensor
    image_tensor = torch.from_numpy(np.array(image)).float()
    
    # Normalize the pixel values to the range [0, 1]
    image_tensor /= 255.0
    
    return image_tensor

In [37]:
def get_data():
    images = []
    labels = []
    print(df.shape[0])
    for i in range(df.shape[0]):
        print(i)
        # 1) récupérer l'UID (dossier)
        uid = df[df.columns[11]][i].split("/")[2]
        folder = Path(images_path) / uid

        jpg_path = folder / os.listdir(folder)[0]  # on prend la première image du dossier

        # 3) ouvrir l'image
        img = Image.open(jpg_path).convert("RGB")
        img = preprocess_image(img)
        images.append(img)

        # 4) label
        if df["pathology"][i] == "MALIGNANT":
            labels.append(1)
        else:
            labels.append(0)

    return images, labels

In [38]:
print(get_data()[0])
print(get_data()[1])

1318
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79


KeyboardInterrupt: 

In [22]:
def create_dataloader(images, labels, batch_size=32):
    # convertir listes -> tensors
    images = torch.stack(images) if isinstance(images, list) else torch.tensor(images)
    labels = torch.tensor(labels, dtype=torch.long)

    dataset = torch.utils.data.TensorDataset(images, labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

In [10]:
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = torch.nn.Linear(32 * 56 * 56, 128)
        self.fc2 = torch.nn.Linear(128, 1)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 56 * 56)
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

In [11]:
model = Model()
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
def train(model, criterion, optimizer, train_loader, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs.squeeze(), labels.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [24]:
train_data = get_data()
train_loader = create_dataloader(train_data[0], train_data[1], batch_size=32)

TypeError: expected Tensor as element 0 in argument 0, but got list

In [14]:
train(model, criterion, optimizer, train_loader, num_epochs=10)

TypeError: object of type 'int' has no len()